In [95]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import List, Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from pydantic import BaseModel
from dotenv import load_dotenv
import os
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [96]:
load_dotenv()
llm = ChatOllama(model="qwen3.5:4b", temperature=0)

In [97]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [98]:
def chat_node(state: ChatState):
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

In [99]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)


graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [100]:
thread_id = '1'


while True:
    query = input("Enter Query here: ")
    print("Human : ",query)

    if query.strip().lower() in ["exit", "quit", "bye"]:
        break
    
    config = {'configurable' : {'thread_id' : thread_id}}
    
    response = chatbot.invoke({"messages": [HumanMessage(content=query)]},config=config)

    print("AI:", response["messages"][-1].content)

Human :  hii my name is aditya who are you
AI: Hi Aditya! It's great to meet you. I'm Qwen3.5, a large language model developed by Tongyi Lab. How can I assist you today?
Human :  what is my name
AI: Your name is Aditya! It's nice to meet you.
Human :  who is ceo of microsoft when he start it
AI: The CEO of Microsoft when it started was **Bill Gates**.

He co-founded the company in 1975 alongside **Paul Allen**. While they were both co-founders and shared leadership roles in the early days, Bill Gates is widely recognized as the first CEO.
Human :  exit
